# Head-to-head on the 26-value grid — this machine, same environment

**Run 1 — NEW `PatternSearchCV`**: requested points, opportunistic poll forced,
default 4-zone bullseye (using fewer zones is expected and fine), MAE objective.

**Run 2 — OLD prototype, bugs intact**: cells 104/106/108/110 of the original
notebook verbatim, with the previously-disclosed two shims (data path, joblib
import) plus ONE toggle inside the execution cell: the active 5-value
`n_estimators` line replaced by the 26-value line that was already present
(commented) in the same cell. The algorithmic code — including the
premature-contraction and non-compounding-pattern-move behaviors — is untouched.

Both runs: same data pipeline, same `TimeSeriesSplit(5)`, same
`neg_mean_absolute_error` objective, same sklearn 1.9, sequential in one kernel
(no core contention). Each run's best point additionally gets a separate
`cross_validate` for CV MAE **and** CV R² (5 extra fits, reported outside the
search cost).

In [1]:
# --- Data pipeline: byte-for-byte replication of the prototype notebook ---
import time
import numpy as np
import pandas as pd

t0 = time.time()
trainBench = pd.read_csv(r"C:\FILES\Code\Benchmarking\train.csv")  # notebook: /home/dell/train.csv

SplitPoint = int(len(trainBench.index) * 0.8)
print("SplitPoint: ", SplitPoint)

df = trainBench

# int64 -> int32; object -> category -> numeric codes (Date included, as in the prototype)
Int64columns = df.select_dtypes(["int64"]).columns
df[Int64columns] = df[Int64columns].astype(np.int32)
cat_columns = df.select_dtypes(["object"]).columns
df[cat_columns] = df[cat_columns].astype("category")
df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)

# the five weather columns the prototype drops (note the dataset's own 'kM' typo)
df = df.drop("Max_Gust_SpeedKm_h", axis=1)
df = df.drop("CloudCover", axis=1)
df = df.drop("Max_VisibilityKm", axis=1)
df = df.drop("Min_VisibilitykM", axis=1)
df = df.drop("Mean_VisibilityKm", axis=1)

# positional 80/20 chronological split
trainBench, validBench = df.iloc[:SplitPoint, :], df.iloc[SplitPoint:, :]
print("Training Set shape", trainBench.shape)
print("Validation Set shape", validBench.shape)
del df

# feature mask: everything but the target (alphabetical order, incl. Date codes
# and NumberOfCustomers, exactly as the prototype had it)
mask = trainBench.columns.difference(["NumberOfSales"])
trainDataset_X = trainBench[mask]
trainDataset_y = trainBench["NumberOfSales"]
validBench_X = validBench[mask]
validBench_y = validBench["NumberOfSales"]
print("Feature Columns:")
print(list(mask))
del trainBench, validBench
print(f"Pipeline done in {time.time() - t0:.1f} s")

SplitPoint:  418416


C:\Users\Home\AppData\Local\Temp\ipykernel_17212\4255920482.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(["object"]).columns


Training Set shape (418416, 31)
Validation Set shape (104605, 31)
Feature Columns:
['AssortmentType', 'Date', 'Events', 'HasPromotions', 'IsHoliday', 'IsOpen', 'Max_Dew_PointC', 'Max_Humidity', 'Max_Sea_Level_PressurehPa', 'Max_TemperatureC', 'Max_Wind_SpeedKm_h', 'Mean_Dew_PointC', 'Mean_Humidity', 'Mean_Sea_Level_PressurehPa', 'Mean_TemperatureC', 'Mean_Wind_SpeedKm_h', 'Min_Dew_PointC', 'Min_Humidity', 'Min_Sea_Level_PressurehPa', 'Min_TemperatureC', 'NearestCompetitor', 'NumberOfCustomers', 'Precipitationmm', 'Region', 'Region_AreaKM2', 'Region_GDP', 'Region_PopulationK', 'StoreID', 'StoreType', 'WindDirDegrees']
Pipeline done in 4.1 s


In [2]:
# ============ RUN 1: NEW PatternSearchCV ============
# 26-value grid (prototype dimension order), opportunistic poll forced,
# default 4-zone bullseye (0.10/0.20/0.50/1.0), MAE scoring.
import os
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from bayes_halving_search_cv import PatternSearchCV

X, y = trainDataset_X, trainDataset_y
N_features = X.shape[1]

def make_clf():
    return ExtraTreesRegressor(n_estimators=240, max_features=max(1, N_features - 15),
                               max_depth=16, n_jobs=1, random_state=0)

param_grid = {
    "max_features": [2, 3, 4],
    "n_estimators": list(range(10, 261, 10)),   # 26 values, the requested points
    "max_depth": [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
}
tscv = TimeSeriesSplit(n_splits=5)

new_search = PatternSearchCV(make_clf(), param_grid,
                             scoring="neg_mean_absolute_error", cv=tscv,
                             n_jobs=-1, poll="opportunistic",
                             random_state=0, verbose=1)
t0 = time.time()
new_search.fit(X, y)
new_elapsed = time.time() - t0
print(f"NEW algorithm search wall-clock: {new_elapsed:.2f} s")


[pattern_search_cv] subsample mode=expanding, zones=[0.1, 0.2, 0.5, 1.0] (rows [41842, 83684, 209208, 418416])


[pattern_search_cv] starts (1): [{'max_features': 3, 'n_estimators': 130, 'max_depth': 11}]


Fitting 5 folds for each of 1 candidates, totalling 5 fits


Fitting 5 folds for each of 1 candidates, totalling 5 fits


Fitting 5 folds for each of 1 candidates, totalling 5 fits


...

Fitting 5 folds for each of 1 candidates, totalling 5 fits


Fitting 5 folds for each of 1 candidates, totalling 5 fits


[pattern_search_cv] climber 0 converged at {'max_features': 4, 'n_estimators': 150, 'max_depth': 17} score=-805.7296842629627


[pattern_search_cv] engine done: 23 evaluations, 15 cache hits, climbers: {0: 'converged'}


NEW algorithm search wall-clock: 824.07 s


In [3]:
# metrics for the new algorithm
res = new_search.cv_results_
new_n_fits = len(res["params"])
new_equiv = float(np.sum(np.asarray(res["n_resources"]) / len(y)))
new_best = new_search.best_params_
new_mae = -new_search.best_score_

# R^2 of the best point (separate cross_validate; NOT counted in search cost)
cvres = cross_validate(make_clf().set_params(**new_best), X, y, cv=tscv,
                       scoring=("r2", "neg_mean_absolute_error"), n_jobs=-1)
new_r2 = cvres["test_r2"].mean()
print("NEW  best:", new_best)
print(f"NEW  fits={new_n_fits}  equiv={new_equiv:.2f}  time={new_elapsed:.1f}s")
print(f"NEW  CV MAE={new_mae:.3f}  CV R2={new_r2:.6f}")
print(f"NEW  zones used: {sorted(set(res['n_resources']))} rows")
print(f"NEW  cache hits: {new_search.n_cache_hits_}")


NEW  best: {'max_features': 4, 'n_estimators': 150, 'max_depth': 17}
NEW  fits=23  equiv=6.80  time=824.1s
NEW  CV MAE=805.730  CV R2=0.809981
NEW  zones used: [np.int64(41842), np.int64(418416)] rows
NEW  cache hits: 15


## Run 2 — prototype, verbatim (bugs intact), 26-value grid

In [4]:
import time
import warnings
import numpy as np
from numpy.ma import MaskedArray # Correct import for MaskedArray
import pandas as pd
from scipy.stats import rankdata

from sklearn.base import BaseEstimator, is_classifier, clone, MetaEstimatorMixin
from sklearn.model_selection import check_cv # Moved to model_selection
# Internal validation functions - check path stability
from sklearn.model_selection._validation import _fit_and_score, _aggregate_score_dicts
from sklearn.exceptions import NotFittedError
# Parallel and delayed are often exposed via sklearn.utils._joblib now
from joblib import Parallel, delayed  # SHIM: sklearn.utils._joblib removed in sklearn>=1.3
from sklearn.utils import check_random_state
# from sklearn.utils.fixes import MaskedArray # Removed
from sklearn.utils.validation import indexable, check_is_fitted
# from sklearn.utils.metaestimators import if_delegate_has_method # Removed
# Scorer-related functions might be directly in metrics or _scorer
from sklearn.metrics import check_scoring
from sklearn.metrics._scorer import _check_multimetric_scoring # Check this path

from abc import ABCMeta, abstractmethod
from collections import defaultdict
from collections.abc import Mapping, Sequence, Iterable
from functools import partial, reduce
from itertools import product
import operator

In [5]:
# --- Custom BaseSearchCV2 ---
# WARNING: Relying on internal scikit-learn APIs like _fit_and_score is fragile
# and may break without warning in future scikit-learn updates.
class BaseSearchCV2(BaseEstimator, MetaEstimatorMixin, metaclass=ABCMeta):
    """Abstract base class for hyper parameter search with cross-validation.
       Updated attempt for scikit-learn >= 1.0, removing iid and if_delegate_has_method.
       Includes fixes for score_params argument and dictionary output of _fit_and_score.
    """

    @abstractmethod
    def __init__(self, estimator, *, scoring=None, n_jobs=None, # iid removed
                 refit=True, cv='warn', verbose=0, pre_dispatch='2*n_jobs',
                 error_score='raise', # Changed default from 'raise-deprecating'
                 return_train_score=True):

        self.scoring = scoring
        self.estimator = estimator
        self.n_jobs = n_jobs
        # self.iid = iid # Removed iid
        self.refit = refit
        self.cv = cv
        self.verbose = verbose
        self.pre_dispatch = pre_dispatch
        self.error_score = error_score
        self.return_train_score = return_train_score

    @property
    def _estimator_type(self):
        return self.estimator._estimator_type

    def score(self, X, y=None):
        """Returns the score on the given data, if the estimator has been refit."""
        self._check_is_fitted('score')
        # Use self.scorer_ which is set during fit
        if not hasattr(self, 'scorer_') or self.scorer_ is None:
             # Fallback to estimator's score method if scorer_ isn't set (should be set in fit)
             if hasattr(self.best_estimator_, 'score'):
                 return self.best_estimator_.score(X, y)
             else:
                 raise ValueError("No score function explicitly defined, "
                                  "and the estimator doesn't provide one.")

        # Handle multi-metric vs single-metric scoring
        # self.scorer_ is the callable (single) or dict (multi)
        score_func = self.scorer_[self.refit] if self.multimetric_ else self.scorer_
        return score_func(self.best_estimator_, X, y)


    def _check_is_fitted(self, method_name):
        if not self.refit:
            raise NotFittedError(f'This {type(self).__name__} instance was initialized '
                                 'with refit=False. {method_name} is '
                                 'available only after refitting on the best '
                                 'parameters. You can refit an estimator '
                                 'manually using the ``best_params_`` attribute')
        else:
            # Ensure 'best_estimator_' attribute exists
            check_is_fitted(self, 'best_estimator_')

    # Methods delegating to best_estimator_
    # Decorators @if_delegate_has_method removed
    def predict(self, X):
        """Call predict on the estimator with the best found parameters."""
        self._check_is_fitted('predict')
        return self.best_estimator_.predict(X)

    def predict_proba(self, X):
        """Call predict_proba on the estimator with the best found parameters."""
        self._check_is_fitted('predict_proba')
        return self.best_estimator_.predict_proba(X)

    def predict_log_proba(self, X):
        """Call predict_log_proba on the estimator with the best found parameters."""
        self._check_is_fitted('predict_log_proba')
        return self.best_estimator_.predict_log_proba(X)

    def decision_function(self, X):
        """Call decision_function on the estimator with the best found parameters."""
        self._check_is_fitted('decision_function')
        return self.best_estimator_.decision_function(X)

    def transform(self, X):
        """Call transform on the estimator with the best found parameters."""
        self._check_is_fitted('transform')
        return self.best_estimator_.transform(X)

    def inverse_transform(self, Xt):
        """Call inverse_transform on the estimator with the best found params."""
        self._check_is_fitted('inverse_transform')
        return self.best_estimator_.inverse_transform(Xt)

    @property
    def classes_(self):
        self._check_is_fitted("classes_")
        return self.best_estimator_.classes_

    @abstractmethod
    def _run_search(self, evaluate_candidates):
        """Mandatory method to be implemented by subclasses."""
        raise NotImplementedError("_run_search not implemented.")

    def fit(self, X, y=None, *, groups=None, **fit_params):
        """Run fit with all sets of parameters."""
        estimator = self.estimator
        # Use default cv=5 if 'warn'
        cv = 5 if self.cv == 'warn' else self.cv
        cv = check_cv(cv, y, classifier=is_classifier(estimator))

        # --- SCORER SETUP (Final Revision) ---
        scoring_input = self.scoring
        if scoring_input is None:
             if hasattr(self.estimator, "score"):
                 _scorer_arg_for_fit_and_score = check_scoring(self.estimator, scoring=None)
                 scorer_name = "score" # Default name when using estimator's score
                 scorers_dict = {scorer_name: _scorer_arg_for_fit_and_score}
                 self.multimetric_ = False
                 self.scorer_ = _scorer_arg_for_fit_and_score
                 refit_metric = scorer_name
             else:
                 raise ValueError("If scoring is None, the estimator must have a score method.")
        elif isinstance(scoring_input, str) or callable(scoring_input):
             # Single metric case
             try:
                 single_scorer_callable = check_scoring(self.estimator, scoring=scoring_input)
                 _scorer_arg_for_fit_and_score = single_scorer_callable

                 # --- Get the actual scorer name ---
                 if isinstance(scoring_input, str):
                      scorer_name = scoring_input
                 else: # It's a callable, use a default name
                      scorer_name = "score"
                 # --- Use scorer_name as the key and for refit_metric ---
                 scorers_dict = {scorer_name: single_scorer_callable}
                 self.multimetric_ = False
                 self.scorer_ = single_scorer_callable
                 refit_metric = scorer_name # <-- Use actual name

             except Exception as e:
                 print(f"Error during single metric scoring setup ('{scoring_input}'): {e}")
                 raise e
        else:
             # Multi-metric case (list, tuple, dict)
             try:
                 scorers_dict, self.multimetric_ = _check_multimetric_scoring(
                     self.estimator, scoring=scoring_input
                 )
                 _scorer_arg_for_fit_and_score = scorers_dict
                 self.scorer_ = scorers_dict
                 # Check refit validity for multi-metric
                 if self.refit is not False and (
                         not isinstance(self.refit, str) or
                         self.refit not in scorers_dict) and not callable(self.refit):
                     raise ValueError("For multi-metric scoring, the parameter refit must be set "
                                      "to a scorer key or a callable. If refit is not needed, "
                                      "set refit=False. %r was passed." % self.refit)
                 refit_metric = self.refit # Use self.refit directly here
             except Exception as e:
                 print(f"Error during multi-metric scoring setup: {e}")
                 raise e
        # --- END SCORER SETUP ---


        X, y, groups = indexable(X, y, groups)
        n_splits = cv.get_n_splits(X, y, groups)

        base_estimator = clone(self.estimator)

        parallel = Parallel(n_jobs=self.n_jobs, verbose=self.verbose,
                            pre_dispatch=self.pre_dispatch)

        # Pass the determined scorer argument (callable or dict) to _fit_and_score
        fit_and_score_kwargs = dict(scorer=_scorer_arg_for_fit_and_score, # Correct arg type passed
                                    fit_params=fit_params,
                                    return_train_score=self.return_train_score,
                                    return_n_test_samples=True,
                                    return_times=True,
                                    return_parameters=False,
                                    error_score=self.error_score,
                                    verbose=self.verbose,
                                    score_params=None)

        results = {}
        all_candidate_params = []
        all_out = []

        # Use a context manager for Parallel
        with parallel:
            def evaluate_candidates(candidate_params):
                candidate_params = list(candidate_params)
                n_candidates = len(candidate_params)

                if self.verbose > 0:
                    print(f"Fitting {n_splits} folds for each of {n_candidates} candidates, "
                          f"totalling {n_candidates * n_splits} fits")

                try:
                    out = parallel(delayed(_fit_and_score)(clone(base_estimator),
                                                           X, y,
                                                           train=train, test=test,
                                                           parameters=parameters,
                                                           **fit_and_score_kwargs)
                                   for parameters, (train, test)
                                   in product(candidate_params,
                                              cv.split(X, y, groups)))
                except ImportError:
                     raise ImportError("Failed to import or use internal function "
                                       "'_fit_and_score'. This custom SearchCV implementation "
                                       "is likely incompatible with the current scikit-learn version.")
                except Exception as e:
                     print(f"Error during parallel execution of _fit_and_score: {e}")
                     raise e


                if not out:
                    raise ValueError('No fits were performed. Was the CV iterator empty? '
                                     'Were there no candidates?')
                elif len(out) != n_candidates * n_splits:
                     print(f"Warning: Expected {n_candidates * n_splits} results, but got {len(out)}. "
                           "This might be due to error_score='raise'.")

                all_candidate_params.extend(candidate_params)
                all_out.extend(out)

                nonlocal results
                # Pass the 'scorers_dict' (always a dict with correct keys) for formatting results
                results = self._format_results(
                    all_candidate_params, scorers_dict, n_splits, all_out)
                return results

            self._run_search(evaluate_candidates)

        # Post-search processing (ranking, best estimator)
        if not results:
             print("Warning: No results were obtained from the search.")
             return self

        # Determine best candidate based on 'refit_metric' determined earlier
        if self.refit or not self.multimetric_:
            if callable(self.refit):
                try:
                    self.best_index_ = self.refit(results)
                    if not isinstance(self.best_index_, int):
                        raise TypeError('best_index_ returned is not an integer')
                    if not (0 <= self.best_index_ < len(results["params"])):
                        raise IndexError('best_index_ index out of range')
                except Exception as e:
                    print(f"Error calling custom refit function: {e}")
                    raise e
            else:
                # Ensure the refit_metric exists in the results
                mean_test_score_key = f"mean_test_{refit_metric}" # Uses correct refit_metric now
                rank_test_score_key = f"rank_test_{refit_metric}" # Uses correct refit_metric now

                if rank_test_score_key not in results:
                     if not any(k.startswith("rank_test_") for k in results.keys()):
                          print("Warning: No rank keys found in cv_results_. Cannot determine best parameters.")
                          self.cv_results_ = results
                          self.n_splits_ = n_splits
                          return self
                     else:
                          available_rank_keys = [k for k in results.keys() if k.startswith("rank_test_")]
                          raise ValueError(f"Ranking key '{rank_test_score_key}' not found in cv_results_. "
                                           f"Ensure scoring includes '{refit_metric}' or refit is a valid scorer key. "
                                           f"Available rank keys: {available_rank_keys}")


                try:
                    # Find the index where rank is 1 (best)
                    if rank_test_score_key not in results or np.all(np.isnan(results[rank_test_score_key])):
                         print(f"Warning: Rank key '{rank_test_score_key}' missing or all NaN. Attempting fallback using nanargmax on mean score.")
                         self.best_index_ = np.nanargmax(results[mean_test_score_key])
                    else:
                         valid_ranks = results[rank_test_score_key][~np.isnan(results[rank_test_score_key])]
                         if 1 not in valid_ranks:
                              print(f"Warning: Rank 1 not found for metric '{refit_metric}'. Attempting fallback using nanargmax.")
                              self.best_index_ = np.nanargmax(results[mean_test_score_key])
                         else:
                              best_indices = np.where(results[rank_test_score_key] == 1)[0]
                              if len(best_indices) == 0:
                                   print(f"Warning: Rank 1 not found unexpectedly for metric '{refit_metric}'. Attempting fallback using nanargmax.")
                                   self.best_index_ = np.nanargmax(results[mean_test_score_key])
                              else:
                                   self.best_index_ = best_indices[0]

                    self.best_score_ = results[mean_test_score_key][self.best_index_]

                except (IndexError, ValueError) as e:
                     print(f"Warning: Could not find best index for metric '{refit_metric}' even with fallbacks. Error: {e}")
                     if mean_test_score_key in results:
                          try:
                              self.best_index_ = np.nanargmax(results[mean_test_score_key])
                              self.best_score_ = results[mean_test_score_key][self.best_index_]
                              print(f"Final fallback successful: Found best index {self.best_index_} using nanargmax.")
                          except (ValueError, IndexError) as fallback_e:
                              raise ValueError(f"Could not find best index for metric '{refit_metric}'. Check cv_results_. Final fallback error: {fallback_e}")
                     else:
                          raise ValueError(f"Cannot determine best index. Mean score key '{mean_test_score_key}' not found.")
                except KeyError:
                     raise KeyError(f"Metric '{refit_metric}' not found in cv_results_. Available metrics: "
                                    f"{[k for k in results if k.startswith('mean_test_')]}")

            if hasattr(self, 'best_index_') and self.best_index_ < len(results["params"]):
                 self.best_params_ = results["params"][self.best_index_]
            else:
                 print("Warning: Could not determine best parameters due to index issue.")
                 self.best_params_ = None


        # Refit the best estimator on the whole dataset
        if self.refit and hasattr(self, 'best_params_') and self.best_params_ is not None:
            self.best_estimator_ = clone(base_estimator).set_params(**self.best_params_)
            refit_start_time = time.time()
            if y is not None:
                self.best_estimator_.fit(X, y, **fit_params)
            else:
                self.best_estimator_.fit(X, **fit_params)
            refit_end_time = time.time()
            self.refit_time_ = refit_end_time - refit_start_time
        elif self.refit:
             print("Warning: Refit skipped because best parameters could not be determined.")


        # Store final results
        self.cv_results_ = results
        self.n_splits_ = n_splits

        return self

    def _format_results(self, candidate_params, scorers, n_splits, out):
        """Format results from _fit_and_score into cv_results_ dictionary.
           Updated for scikit-learn >= 1.0 where _fit_and_score returns a dict.
        """
        n_candidates = len(candidate_params)

        if not out:
            # print("Warning: _format_results received empty 'out' list.") # Keep warnings minimal
            return {}

        # --- DEBUG PRINTS REMOVED ---

        # --- UNPACKING FROM LIST OF DICTIONARIES ---
        train_score_dicts_list = []
        test_score_dicts_list = []
        test_sample_counts_list = []
        fit_time_list = []
        score_time_list = []
        processed_elements = 0

        expected_tuple_size = 5 if self.return_train_score else 4 # Still useful for context in warnings

        for i, result_dict in enumerate(out):
            if isinstance(result_dict, dict):
                processed_elements += 1
                try:
                    test_scores = result_dict.get('test_scores', {})
                    if not isinstance(test_scores, dict):
                         scorer_key = list(scorers.keys())[0] if len(scorers) == 1 else 'score'
                         test_scores = {scorer_key: test_scores}

                    if self.return_train_score:
                        train_scores = result_dict.get('train_scores', {})
                        if not isinstance(train_scores, dict):
                             scorer_key = list(scorers.keys())[0] if len(scorers) == 1 else 'score'
                             train_scores = {scorer_key: train_scores}
                        train_score_dicts_list.append(train_scores)

                    n_test_samples = result_dict.get('n_test_samples', np.nan)
                    fit_time_val = result_dict.get('fit_time', np.nan)
                    score_time_val = result_dict.get('score_time', np.nan)

                    test_score_dicts_list.append(test_scores)
                    test_sample_counts_list.append(n_test_samples)
                    fit_time_list.append(fit_time_val)
                    score_time_list.append(score_time_val)

                    fit_error = result_dict.get('fit_error', None)
                    if fit_error is not None:
                         print(f"Warning: Fit error reported for element {i}: {fit_error}")

                except Exception as e:
                     print(f"Warning: Error processing result dictionary element {i}: {e}. Value: {result_dict}. Skipping.")

            else:
                print(f"ERROR: Element {i} in 'out' is not a dictionary. "
                      f"Got Type: {type(result_dict)}, Value: {str(result_dict)[:500]}... Skipping.")
        # --- END UNPACKING ---

        expected_results = n_candidates * n_splits
        if processed_elements == 0:
             print("ERROR: No valid results could be unpacked from 'out'. Cannot format results.")
             return {}
        elif processed_elements < expected_results:
             print(f"Warning: Only {processed_elements}/{expected_results} results were successfully unpacked. "
                   "This might indicate errors during fitting/scoring for some folds/candidates.")

        # --- AGGREGATION AND STORAGE ---
        try:
            test_scores_agg = _aggregate_score_dicts(test_score_dicts_list)
            if self.return_train_score and train_score_dicts_list:
                train_scores_agg = _aggregate_score_dicts(train_score_dicts_list)
            else:
                train_scores_agg = None
        except Exception as e:
             print(f"Error during score aggregation: {e}")
             return {}

        results = {}

        # Helper function _store
        def _store(key_name, array_list, n_splits_effective, n_candidates_effective, weights=None, splits=False, rank=False):
            """Store metrics in the results dictionary. Adjusted for potentially fewer results."""
            orig_n_candidates = len(candidate_params)
            orig_n_splits = n_splits

            is_empty = False
            if array_list is None: is_empty = True
            elif isinstance(array_list, (list, np.ndarray)):
                current_size = array_list.size if isinstance(array_list, np.ndarray) else len(array_list)
                if current_size == 0: is_empty = True
            else:
                 # print(f"Warning: Unexpected type for array_list in _store for key '{key_name}': {type(array_list)}. Treating as empty.")
                 is_empty = True

            if is_empty:
                 # print(f"Warning: Empty list/array provided for key '{key_name}'. Filling results with NaN.")
                 results[f'mean_{key_name}'] = np.full(orig_n_candidates, np.nan)
                 results[f'std_{key_name}'] = np.full(orig_n_candidates, np.nan)
                 if rank: results[f"rank_{key_name}"] = np.full(orig_n_candidates, np.nan)
                 if splits:
                     for split_i in range(orig_n_splits): results[f"split{split_i}_{key_name}"] = np.full(orig_n_candidates, np.nan)
                 return

            try:
                if not isinstance(array_list, np.ndarray): array_list = np.array(array_list, dtype=np.float64)
                array = array_list.reshape(n_candidates_effective, n_splits_effective)
            except ValueError as e:
                # print(f"ERROR: Could not reshape array for key '{key_name}'. "
                #       f"Expected {n_candidates_effective * n_splits_effective} elements from {array_list.size} valid results. Error: {e}. Filling with NaN.")
                results[f'mean_{key_name}'] = np.full(orig_n_candidates, np.nan)
                results[f'std_{key_name}'] = np.full(orig_n_candidates, np.nan)
                if rank: results[f"rank_{key_name}"] = np.full(orig_n_candidates, np.nan)
                if splits:
                    for split_i in range(orig_n_splits): results[f"split{split_i}_{key_name}"] = np.full(orig_n_candidates, np.nan)
                return

            with warnings.catch_warnings():
                 warnings.simplefilter("ignore", category=RuntimeWarning)
                 array_means_effective = np.ma.average(np.ma.masked_invalid(array), axis=1, weights=weights).filled(np.nan)
                 array_stds_effective = np.nanstd(array, axis=1)

            array_means = np.full(orig_n_candidates, np.nan)
            array_stds = np.full(orig_n_candidates, np.nan)
            fill_len = min(orig_n_candidates, n_candidates_effective)
            array_means[:fill_len] = array_means_effective[:fill_len]
            array_stds[:fill_len] = array_stds_effective[:fill_len]

            results[f'mean_{key_name}'] = array_means
            results[f'std_{key_name}'] = array_stds

            if splits:
                 for split_i in range(orig_n_splits):
                     split_data = np.full(orig_n_candidates, np.nan)
                     if split_i < n_splits_effective:
                         if array.shape[1] > split_i: split_data[:fill_len] = array[:fill_len, split_i]
                     results[f"split{split_i}_{key_name}"] = split_data

            if rank:
                try:
                    # For any scikit-learn scorer, a higher value is better.
                    # To give rank 1 to the highest score, we rank the negated array of scores.
                    rank_scores = -array_means
                    results[f"rank_{key_name}"] = rankdata(rank_scores, method='min').astype(int)
                except ValueError:
                    print(f"Warning: Could not compute rank for key '{key_name}'.")
                    results[f"rank_{key_name}"] = np.full(orig_n_candidates, np.nan)

        # Determine effective dimensions
        if processed_elements == 0: n_candidates_effective, n_splits_effective = 0, 0
        elif processed_elements % n_splits == 0:
             n_candidates_effective = processed_elements // n_splits
             n_splits_effective = n_splits
        elif processed_elements < n_candidates * n_splits:
             n_candidates_effective = n_candidates
             n_splits_effective = n_splits
             # print(f"DEBUG: Assuming {n_candidates_effective} candidates and {n_splits_effective} splits for reshaping, based on {processed_elements} results.")
        else: n_candidates_effective, n_splits_effective = n_candidates, n_splits

        _store('fit_time', fit_time_list, n_splits_effective, n_candidates_effective)
        _store('score_time', score_time_list, n_splits_effective, n_candidates_effective)

        param_results = defaultdict(lambda: MaskedArray(np.empty(n_candidates,), mask=True, dtype=object))
        for cand_i, params in enumerate(candidate_params):
            for name, value in params.items(): param_results[f"param_{name}"][cand_i] = value
        results.update(param_results)
        results['params'] = candidate_params

        try:
            if len(test_sample_counts_list) == n_candidates_effective * n_splits_effective:
                 test_sample_counts_arr = np.array(test_sample_counts_list).reshape(n_candidates_effective, n_splits_effective)
                 weights = test_sample_counts_arr[0]
            else: weights = None
        except ValueError as e: weights = None

        for scorer_name in scorers.keys():
            if scorer_name in test_scores_agg:
                 _store(f'test_{scorer_name}', test_scores_agg[scorer_name], n_splits_effective, n_candidates_effective, splits=True, rank=True, weights=None)
            else:
                 # print(f"Warning: Test score for '{scorer_name}' not found in aggregated results.")
                 results[f'mean_test_{scorer_name}'] = np.full(n_candidates, np.nan)
                 results[f'std_test_{scorer_name}'] = np.full(n_candidates, np.nan)
                 results[f'rank_test_{scorer_name}'] = np.full(n_candidates, np.nan)
                 for split_i in range(n_splits): results[f"split{split_i}_test_{scorer_name}"] = np.full(n_candidates, np.nan)

            if self.return_train_score and train_scores_agg:
                 if scorer_name in train_scores_agg:
                     _store(f'train_{scorer_name}', train_scores_agg[scorer_name], n_splits_effective, n_candidates_effective, splits=True, weights=None)
                 else:
                     # print(f"Warning: Train score for '{scorer_name}' not found in aggregated results.")
                     results[f'mean_train_{scorer_name}'] = np.full(n_candidates, np.nan)
                     results[f'std_train_{scorer_name}'] = np.full(n_candidates, np.nan)
                     for split_i in range(n_splits): results[f"split{split_i}_train_{scorer_name}"] = np.full(n_candidates, np.nan)

        return results

In [6]:
# --- Custom PatternSearchCV ---
class PatternSearchCV(BaseSearchCV2):
    _required_parameters = ["estimator", "param_distributions"]

    # Updated __init__ signature and super() call
    def __init__(self, estimator, param_distributions, *, scoring=None,
                 n_jobs=None, refit=True,
                 cv='warn', verbose=0, pre_dispatch='2*n_jobs',
                 random_state=None, error_score='raise',
                 return_train_score=False):

        self.param_distributions = param_distributions
        self.random_state = random_state
        super().__init__(
            estimator=estimator, scoring=scoring,
            n_jobs=n_jobs, refit=refit, cv=cv, verbose=verbose,
            pre_dispatch=pre_dispatch, error_score=error_score,
            return_train_score=return_train_score)

        # --- Inner Dimension Class --- (No changes needed here)
        class Dimension():
            def __init__(self, value):
                if isinstance(value,tuple):
                   lower, upper, length = value[0], value[1], value[2]
                   value=[lower + x*(upper-lower)/(length-1) for x in range(int(length))]
                   if all(isinstance(v, int) or (isinstance(v, float) and v.is_integer()) for v in [lower, upper]):
                       value = [int(round(v)) for v in value]
                if not isinstance(value, list):
                    try: value = list(value)
                    except TypeError: raise TypeError(f"Dimension input must be a list, tuple, or iterable. Got {type(value)}")
                self.value=value
                self.value.sort()
                self.min=self.value[0] if self.value else None
                self.max=self.value[-1] if self.value else None
                self.midptidx=max(0, int(len(self.value)/2 - 0.5))
                self.midpoint=self.value[self.midptidx] if self.value else None
                self.Delta=max(0, int((len(value)-1)-self.midptidx))
                self.BestValue=self.midpoint
                self.CurrIndex=self.midptidx
                self.BestIndex=self.midptidx
        # --- End Inner Dimension Class ---

        # Initialize Search Space
        self.Space={}
        print("Pattern Search Dimensions:")
        param_cols = []
        param_dtypes = {}
        for Dkey, Dval in self.param_distributions.items():
            try:
                self.Space[Dkey]=Dimension(Dval)
                param_cols.append(Dkey)
                dim_values = self.Space[Dkey].value
                if dim_values and all(isinstance(v, int) or (isinstance(v, np.integer)) for v in dim_values):
                     param_dtypes[Dkey] = int
                else:
                     param_dtypes[Dkey] = float
                print(f"  {Dkey}: {self.Space[Dkey].value}")
            except Exception as e:
                print(f"Error initializing dimension '{Dkey}' with value {Dval}: {e}")
                raise

        # --- Initialize ResultDf with POSITIVE score column name ---
        # Determine the base scorer name (used internally by BaseSearchCV2)
        self.internal_score_name = self.refit if isinstance(self.refit, str) else 'score'
        # Determine the display name (remove "neg_")
        self.display_score_name = self.internal_score_name.replace("neg_", "")
        # Check if the name actually changed (i.e., if it was a negated metric)
        self.lower_score_is_better = self.internal_score_name.startswith("neg_")

        df_cols = param_cols + [self.display_score_name] # Use display name
        param_dtypes[self.display_score_name] = float # Score is float
        self.ResultDf = pd.DataFrame({col: pd.Series(dtype=dtype) for col, dtype in param_dtypes.items()})
        self.ResultDf = self.ResultDf[df_cols] # Ensure column order
        # --- End ResultDf Initialization ---

    def _run_search(self, evaluate_candidates):
        """Search best parameters using Pattern Search Method. Displays positive errors."""
        alfa=2
        k=0
        Ndimensions=len(self.Space)
        param_cols = list(self.Space.keys())

        xc={}
        for Dkey, Dval in self.Space.items():
            if Dval.midpoint is None: raise ValueError(f"Dimension '{Dkey}' is empty or invalid.")
            xc[Dkey]=Dval.midpoint

        DeltaVector=[]
        for CurDim in range(Ndimensions): DeltaVector.append(list(self.Space.values())[CurDim].Delta)
        print(" ")
        print("Current Delta values for each dimension: ", DeltaVector)
        print(" ")

        print(xc)
        initial_results = evaluate_candidates([xc])

        # Key to access score in the results dict from evaluate_candidates
        internal_score_key = f"mean_test_{self.internal_score_name}"

        if internal_score_key not in initial_results:
             available_keys = list(initial_results.keys())
             if not available_keys: raise RuntimeError("Initial evaluation failed to produce any results.")
             raise KeyError(f"Internal score key '{internal_score_key}' not found in results after initial evaluation. Available keys: {available_keys}")

        # BestScore holds the internal (potentially negated) value for comparison
        BestScore = initial_results[internal_score_key][-1]

        xd = xc.copy()
        # Store the DISPLAY value (positive error or original score)
        display_value = abs(BestScore) if self.lower_score_is_better else BestScore
        xd.update({self.display_score_name: display_value})
        self.ResultDf = pd.concat([self.ResultDf, pd.DataFrame([xd])], ignore_index=True)
        print(self.ResultDf)

        BestIdx = 0
        InitialExplorationIdx = 0
        i=0
        Continue=0

        while Continue < 3:
            ExplorationSuccessful = False
            i = 0
            while i < Ndimensions:
                CurrentDimKey = list(self.Space.keys())[i]
                CurrentDimObj = list(self.Space.values())[i]

                if not CurrentDimObj.value: i += 1; continue

                CurrentBestValInDim = xc[CurrentDimKey]
                try: CurrentBestIdxInDim = CurrentDimObj.value.index(CurrentBestValInDim)
                except ValueError: CurrentBestIdxInDim = CurrentDimObj.midptidx

                for Direction in [1, -1]:
                    xn = xc.copy()
                    StepSize = CurrentDimObj.Delta
                    if StepSize == 0: continue

                    NewIndex = CurrentBestIdxInDim + Direction * StepSize
                    NewIndex = max(0, min(len(CurrentDimObj.value) - 1, NewIndex))

                    if NewIndex != CurrentBestIdxInDim:
                        NewValue = CurrentDimObj.value[NewIndex]
                        xn[CurrentDimKey] = NewValue

                        already_evaluated = False
                        # (Comparison logic remains the same)
                        for row_idx in self.ResultDf.index:
                             row_params = self.ResultDf.loc[row_idx, param_cols].to_dict()
                             match = True
                             for p in param_cols:
                                 xn_val = xn[p]; row_val = row_params[p]
                                 if isinstance(xn_val, np.integer): xn_val = int(xn_val)
                                 if isinstance(row_val, np.integer): row_val = int(row_val)
                                 if isinstance(xn_val, np.floating): xn_val = float(xn_val)
                                 if isinstance(row_val, np.floating): row_val = float(row_val)
                                 if isinstance(xn_val, float):
                                     if not np.isclose(row_val, xn_val): match = False; break
                                 elif row_val != xn_val: match = False; break
                             if match: already_evaluated = True; break

                        if not already_evaluated:
                            k += 1
                            print(Direction, xn)
                            current_results = evaluate_candidates([xn])

                            if not current_results or internal_score_key not in current_results:
                                 available_keys = list(current_results.keys()) if current_results else "None"
                                 print(f"Warning: Internal score key '{internal_score_key}' not found in results for point {xn}. Available keys: {available_keys}. Skipping point.")
                                 continue

                            # CurrScore holds the internal (potentially negated) value
                            CurrScore = current_results[internal_score_key][-1]

                            xd = xn.copy()
                            # Store the DISPLAY value
                            display_value = abs(CurrScore) if self.lower_score_is_better else CurrScore
                            xd.update({self.display_score_name: display_value})
                            self.ResultDf = pd.concat([self.ResultDf, pd.DataFrame([xd])], ignore_index=True)
                            print(self.ResultDf)

                            # --- Compare internal (negated) scores ---
                            # Higher negated score is better
                            score_improved = False
                            if np.isnan(BestScore):
                                 if not np.isnan(CurrScore): score_improved = True
                            elif not np.isnan(CurrScore):
                                 if CurrScore > BestScore: score_improved = True # Always compare internal scores
                            # --- End comparison ---

                            if score_improved:
                                BestScore = CurrScore # Update BestScore with the internal value
                                xc = xn.copy()
                                BestIdx = len(self.ResultDf) - 1
                                ExplorationSuccessful = True
                                break
                i += 1

            # --- Pattern Move ---
            if ExplorationSuccessful:
                # (Theoretical move calculation remains the same)
                params_at_best = self.ResultDf.loc[BestIdx, param_cols].to_dict()
                params_at_initial = self.ResultDf.loc[InitialExplorationIdx, param_cols].to_dict()
                xp_theoretical = {}; valid_theoretical = True
                for p in param_cols:
                    try: xp_theoretical[p] = 2 * params_at_best[p] - params_at_initial[p]
                    except TypeError: valid_theoretical = False; break
                if not valid_theoretical: ExplorationSuccessful = False; print("..."); continue
                print("Theoretical Pattern Move: ", list(xp_theoretical.values()))

                # (Nearest point calculation remains the same)
                xn_pattern = {}; valid_pattern_point = True
                for n, Dkey in enumerate(param_cols):
                    Dval = self.Space[Dkey]
                    if not Dval.value: valid_pattern_point = False; break
                    target_val = xp_theoretical[Dkey]
                    try: closest_val = min(Dval.value, key=lambda x: abs(x - target_val)); xn_pattern[Dkey] = closest_val
                    except (TypeError, ValueError) as e: valid_pattern_point = False; break
                if not valid_pattern_point: ExplorationSuccessful = False; print("..."); continue
                xn_pattern_print = {k: (int(v) if isinstance(v, np.integer) else v) for k,v in xn_pattern.items()}
                print("Nearest Pattern move point", xn_pattern_print)

                # (Check if evaluated remains the same)
                already_evaluated_pattern = False
                for row_idx in self.ResultDf.index:
                     row_params = self.ResultDf.loc[row_idx, param_cols].to_dict()
                     match = True
                     for p in param_cols:
                         xn_val = xn_pattern[p]; row_val = row_params[p]
                         if isinstance(xn_val, np.integer): xn_val = int(xn_val)
                         if isinstance(row_val, np.integer): row_val = int(row_val)
                         if isinstance(xn_val, np.floating): xn_val = float(xn_val)
                         if isinstance(row_val, np.floating): row_val = float(row_val)
                         if isinstance(xn_val, float):
                             if not np.isclose(row_val, xn_val): match = False; break
                         elif row_val != xn_val: match = False; break
                     if match: already_evaluated_pattern = True; break

                if not already_evaluated_pattern:
                    k += 1
                    print(xn_pattern_print)
                    print("Executing Pattern Move...")
                    current_results = evaluate_candidates([xn_pattern])

                    if not current_results or internal_score_key not in current_results:
                         available_keys = list(current_results.keys()) if current_results else "None"
                         print(f"Warning: Internal score key '{internal_score_key}' not found in results for pattern point {xn_pattern}. Available keys: {available_keys}. Skipping point.")
                         ExplorationSuccessful = False
                    else:
                        # PatternScore holds the internal (potentially negated) value
                        PatternScore = current_results[internal_score_key][-1]
                        xd = xn_pattern.copy()
                        # Store the DISPLAY value
                        display_value = abs(PatternScore) if self.lower_score_is_better else PatternScore
                        xd.update({self.display_score_name: display_value})
                        self.ResultDf = pd.concat([self.ResultDf, pd.DataFrame([xd])], ignore_index=True)
                        print(self.ResultDf)

                        # --- Compare internal (negated) scores ---
                        score_improved = False
                        if np.isnan(BestScore):
                             if not np.isnan(PatternScore): score_improved = True
                        elif not np.isnan(PatternScore):
                             if PatternScore > BestScore: score_improved = True # Always compare internal scores
                        # --- End comparison ---

                        if score_improved:
                            BestScore = PatternScore # Update BestScore with internal value
                            xc = xn_pattern.copy()
                            BestIdx = len(self.ResultDf) - 1
                            InitialExplorationIdx = BestIdx
                            Continue = 0
                            ExplorationSuccessful = True
                        else:
                            ExplorationSuccessful = False
                else:
                    print(f"Nearest Pattern move point {xn_pattern_print} has been evaluated")
                    ExplorationSuccessful = False

            # --- Delta Reduction ---
            if not ExplorationSuccessful:
                DeltaVector = []
                for CurDim in range(Ndimensions):
                    Dobj = list(self.Space.values())[CurDim]
                    if Dobj.Delta > 0: Dobj.Delta = max(1, int(0.5 + Dobj.Delta / alfa))
                    DeltaVector.append(Dobj.Delta)
                print("Current Delta values for each dimension: ", DeltaVector)

            # Check for termination condition
            if all(d <= 1 for d in DeltaVector): Continue += 1
            else: Continue = 0

        print(" ")
        print("--- Pattern Search Finished ---")
        print(f"Total Fits: {k}")
        print(" ")
        print("Best Parameters Found:")
        # --- Find best row in ResultDf using display name and idxmin/idxmax ---
        if self.ResultDf.empty or self.ResultDf[self.display_score_name].isnull().all():
             print("  No valid scores found during search.")
        else:
             if self.lower_score_is_better:
                 best_row_idx = self.ResultDf[self.display_score_name].idxmin() # Find minimum positive error
             else:
                 best_row_idx = self.ResultDf[self.display_score_name].idxmax() # Find maximum positive score

             best_row = self.ResultDf.loc[best_row_idx]
             print(best_row)
        print(" ")



In [7]:
t_old_start = time.time()

In [8]:
# --- Execution Block ---
from sklearn.metrics import r2_score, mean_absolute_error
# Ensure X, y are defined from the preprocessing steps above
if 'trainDataset_X' not in globals() or 'trainDataset_y' not in globals():
     print("Error: Training data (trainDataset_X, trainDataset_y) not prepared. Exiting.")
     sys.exit(1)

# Use the full training data for the search as per original code intent
X, y = trainDataset_X, trainDataset_y
print(f"Using full training set for Pattern Search: X shape {X.shape}, y shape {y.shape}")


# Build model (ensure parameters are valid for the chosen regressor)
N_features = X.shape[1]
max_features_default = max(1, N_features - 15) # Ensure >= 1

try:
    clf = ExtraTreesRegressor(n_estimators=240, # Default, might be overridden by search
                              max_features=max_features_default, # Default
                              max_depth=16, # Default
                              n_jobs=1,
                              random_state=0)
    # Test if parameters are valid
    clf.get_params()
    print("ExtraTreesRegressor initialized successfully.")
except TypeError as e:
    print(f"Error initializing ExtraTreesRegressor. Check default parameters: {e}")
    # Fallback or exit
    clf = ExtraTreesRegressor(n_jobs=-1, random_state=0) # Minimal init
    print("Initialized with minimal parameters.")


# Specify parameters and distributions for Pattern Search
# Ensure values are appropriate for the parameters

#######################DEFINE PARAMETERS TO BE SEARCHED HERE ##########################################
param_dist={
    'max_features': [2, 3, 4], # Example discrete list
    # Linspace for n_estimators: start, stop, num_points
    # Ensure start > 0 for n_estimators
    
    'n_estimators': list(np.linspace(0, 250, num=26, endpoint=True,dtype=int)+10), # GRID TOGGLE: activated the cell's own commented 26-value line
    # Range for max_depth: start, stop (exclusive), step
    'max_depth': list(range(5, 18, 1)) # Example range with step
}

# Utility function to report best scores (Modified for positive error display)
def report(results, n_top=3):
    # Determine the score key used (based on refit or default 'score')
    refit_metric = 'score' # Default
    # Check if pattern_search exists and has refit attribute
    # Use globals().get() for safer access in case pattern_search doesn't exist yet
    search_obj = globals().get('pattern_search')
    if search_obj and hasattr(search_obj, 'refit') and isinstance(search_obj.refit, str):
        refit_metric = search_obj.refit
    elif search_obj and hasattr(search_obj, 'scoring') and isinstance(search_obj.scoring, str):
         refit_metric = search_obj.scoring

    # These keys are based on the actual keys in cv_results_ (still negated)
    internal_score_key = f"mean_test_{refit_metric}"
    internal_std_key = f"std_test_{refit_metric}"
    internal_rank_key = f"rank_test_{refit_metric}"

    # Determine if the metric is a negated error for display purposes
    lower_is_better = refit_metric.startswith("neg_")

    if internal_rank_key not in results:
        print(f"Warning: Rank key '{internal_rank_key}' not found in results. Cannot report ranks.")
        # Fallback reporting based on sorting score
        if internal_score_key in results:
            print(f"Reporting top {n_top} based on sorting '{internal_score_key}':")
            scores_for_sort = np.array(results[internal_score_key], dtype=float)
            # Sort ascending for lower_is_better, descending otherwise
            sorted_indices = np.argsort(scores_for_sort)
            if not lower_is_better: sorted_indices = sorted_indices[::-1]

            num_valid_scores = np.sum(~np.isnan(scores_for_sort))
            for i in range(min(n_top, num_valid_scores)):
                candidate = sorted_indices[i]
                print(f"Rank {i+1} (by score):")
                mean_score = results[internal_score_key][candidate]
                std_score = results.get(internal_std_key, [np.nan]*len(results[internal_score_key]))[candidate]
                # --- Display positive score ---
                display_score = abs(mean_score) if lower_is_better else mean_score
                lower_better_str = " (lower is better)" if lower_is_better else ""
                print(f"Mean validation score: {display_score:.3f}{lower_better_str} (std: {std_score:.3f})")
                # --- End display positive score ---
                print(f"Parameters: {results['params'][candidate]}")
                print("")
        else:
            print("Error: Cannot report results as score keys are missing.")
        return

    # Original reporting using ranks (ranks are based on internal negated scores)
    print(f"Reporting top {n_top} based on '{internal_rank_key}':")
    for i in range(1, n_top + 1):
        try:
            valid_ranks = results[internal_rank_key][~np.isnan(results[internal_rank_key])] if internal_rank_key in results else []
            candidates = np.flatnonzero(results[internal_rank_key] == i)

            if candidates.size == 0:
                 if i == 1 and len(valid_ranks) > 0: print(f"Warning: No candidate found with rank {i}. Check results.")
                 continue

            for candidate in candidates:
                print(f"Model with rank: {i}")
                mean_score = results[internal_score_key][candidate] # Get internal score
                std_score = results.get(internal_std_key, [np.nan]*len(results[internal_score_key]))[candidate]
                # --- Display positive score ---
                display_score = abs(mean_score) if lower_is_better else mean_score
                lower_better_str = " (lower is better)" if lower_is_better else ""
                print(f"Mean validation score: {display_score:.3f}{lower_better_str} (std: {std_score:.3f})")
                # --- End display positive score ---
                print(f"Parameters: {results['params'][candidate]}")
                print("")
        except KeyError as e: print(f"Error accessing results key: {e}. Cannot report fully."); break
        except IndexError as e: print(f"Error accessing results index: {e}. Results might be corrupt."); break

# Setup Cross-Validation splitter
print("Using Time Series Cross Validation for Pattern Search")
# Ensure n_splits is feasible
n_splits_tscv_search = min(5, len(X) // 2)
if n_splits_tscv_search < 2:
    print(f"Warning: Dataset size ({len(X)}) too small for TimeSeriesSplit with n_splits={n_splits_tscv_search}. Adjust CV strategy or data.")
    tscv = TimeSeriesSplit(n_splits=max(2, n_splits_tscv_search))
else:
     tscv = TimeSeriesSplit(n_splits=n_splits_tscv_search)


# Run Pattern Search
# Instantiate PatternSearchCV: Use 'neg_mean_absolute_error' for scoring and refit
print("Attempting PatternSearch with scoring='neg_mean_absolute_error'")
pattern_search = PatternSearchCV(clf, param_distributions=param_dist,
                                 scoring='neg_mean_absolute_error', # <-- CHANGED
                                 cv=tscv,
                                 n_jobs=-1,
                                 verbose=0,
                                 refit='neg_mean_absolute_error', # <-- CHANGED
                                 return_train_score=True)

print("\nStarting Pattern Search...")
start = time.time()
try:
    pattern_search.fit(X, y)
    print(f"\nPatternSearchCV took {time.time() - start:.2f} seconds.")

    # Report results if fit completes
    print("\n--- Search Results ---")
    if hasattr(pattern_search, 'cv_results_') and pattern_search.cv_results_:
         report(pattern_search.cv_results_)
    else:
         print("No results generated by Pattern Search.")

    print("\nBest Parameters found by Pattern Search:")
    if hasattr(pattern_search, 'best_params_'):
        print(pattern_search.best_params_)
    else:
        print("Best parameters not determined (fit might have failed or refit=False).")

    print(f"\nBest Score ({pattern_search.display_score_name}):") # Use the display name
    if hasattr(pattern_search, 'best_score_'):
        internal_best_score = pattern_search.best_score_
        # Use the flag set during __init__
        display_best_score = abs(internal_best_score) if pattern_search.lower_score_is_better else internal_best_score
        # Adjust text based on whether lower is better for the *displayed* score
        lower_better_str = " (lower is better)" if pattern_search.lower_score_is_better else ""
        print(f"{display_best_score:.6f}{lower_better_str}")
    else:
        print("Best score not determined.")

    print("\nDataFrame of Evaluated Points (Pattern Search):")
    print(pattern_search.ResultDf)

    print("\nFull cv_results_ DataFrame:")
    if hasattr(pattern_search, 'cv_results_') and pattern_search.cv_results_:
         cv_results_copy = pattern_search.cv_results_.copy()
         if 'params' in cv_results_copy:
             # Use np.integer for checking numpy int types
             cv_results_copy['params'] = [
                 {k: (int(v) if isinstance(v, np.integer) else v) for k, v in p.items()}
                 for p in cv_results_copy['params']
             ]
         results_df = pd.DataFrame(cv_results_copy)
         with pd.option_context('display.max_rows', 100, 'display.max_columns', None):
             print(results_df)
    else:
         print("cv_results_ attribute not found or is empty.")

except Exception as e:
    print(f"\nAn error occurred during Pattern Search: {e}")
    import traceback
    traceback.print_exc()

Using full training set for Pattern Search: X shape (418416, 30), y shape (418416,)
ExtraTreesRegressor initialized successfully.
Using Time Series Cross Validation for Pattern Search
Attempting PatternSearch with scoring='neg_mean_absolute_error'
Pattern Search Dimensions:
  max_features: [2, 3, 4]
  n_estimators: [np.int64(10), np.int64(20), np.int64(30), np.int64(40), np.int64(50), np.int64(60), np.int64(70), np.int64(80), np.int64(90), np.int64(100), np.int64(110), np.int64(120), np.int64(130), np.int64(140), np.int64(150), np.int64(160), np.int64(170), np.int64(180), np.int64(190), np.int64(200), np.int64(210), np.int64(220), np.int64(230), np.int64(240), np.int64(250), np.int64(260)]
  max_depth: [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]

Starting Pattern Search...
 
Current Delta values for each dimension:  [1, 13, 6]
 
{'max_features': 3, 'n_estimators': np.int64(130), 'max_depth': 11}


   max_features  n_estimators  max_depth  mean_absolute_error
0             3           130         11          1128.966638
1 {'max_features': 4, 'n_estimators': np.int64(130), 'max_depth': 11}


   max_features  n_estimators  max_depth  mean_absolute_error
0             3           130         11          1128.966638
1             4           130         11          1009.342784
1 {'max_features': 4, 'n_estimators': np.int64(260), 'max_depth': 11}


   max_features  n_estimators  max_depth  mean_absolute_error
0             3           130         11          1128.966638
1             4           130         11          1009.342784
2             4           260         11          1026.250951
-1 {'max_features': 4, 'n_estimators': np.int64(10), 'max_depth': 11}


   max_features  n_estimators  max_depth  mean_absolute_error
0             3           130         11          1128.966638
1             4           130         11          1009.342784
2             4           260         11          1026.250951
3             4            10         11          1070.391366
1 {'max_features': 4, 'n_estimators': np.int64(130), 'max_depth': 17}


...

    max_features  n_estimators  max_depth  mean_absolute_error
0              3           130         11          1128.966638
1              4           130         11          1009.342784
2              4           260         11          1026.250951
3              4            10         11          1070.391366
4              4           130         17           805.038061
5              3           130         17           907.568961
6              4           200         17           814.572813
7              4            60         17           818.675868
8              4           130         14           887.193431
9              4           170         17           807.968379
10             4            90         17           809.302988
11             4           130         15           863.945454
12             4           150         17           805.729684
13             4           110         17           812.229550
-1 {'max_features': 4, 'n_estimators': np.int64(130), '

    max_features  n_estimators  max_depth  mean_absolute_error
0              3           130         11          1128.966638
1              4           130         11          1009.342784
2              4           260         11          1026.250951
3              4            10         11          1070.391366
4              4           130         17           805.038061
5              3           130         17           907.568961
6              4           200         17           814.572813
7              4            60         17           818.675868
8              4           130         14           887.193431
9              4           170         17           807.968379
10             4            90         17           809.302988
11             4           130         15           863.945454
12             4           150         17           805.729684
13             4           110         17           812.229550
14             4           130         16           831

    max_features  n_estimators  max_depth  mean_absolute_error
0              3           130         11          1128.966638
1              4           130         11          1009.342784
2              4           260         11          1026.250951
3              4            10         11          1070.391366
4              4           130         17           805.038061
5              3           130         17           907.568961
6              4           200         17           814.572813
7              4            60         17           818.675868
8              4           130         14           887.193431
9              4           170         17           807.968379
10             4            90         17           809.302988
11             4           130         15           863.945454
12             4           150         17           805.729684
13             4           110         17           812.229550
14             4           130         16           831

    max_features  n_estimators  max_depth  mean_absolute_error
0              3           130         11          1128.966638
1              4           130         11          1009.342784
2              4           260         11          1026.250951
3              4            10         11          1070.391366
4              4           130         17           805.038061
5              3           130         17           907.568961
6              4           200         17           814.572813
7              4            60         17           818.675868
8              4           130         14           887.193431
9              4           170         17           807.968379
10             4            90         17           809.302988
11             4           130         15           863.945454
12             4           150         17           805.729684
13             4           110         17           812.229550
14             4           130         16           831


PatternSearchCV took 1546.15 seconds.

--- Search Results ---
Reporting top 3 based on 'rank_test_neg_mean_absolute_error':
Model with rank: 1
Mean validation score: 805.038 (lower is better) (std: 22.482)
Parameters: {'max_features': 4, 'n_estimators': np.int64(130), 'max_depth': 17}

Model with rank: 2
Mean validation score: 805.730 (lower is better) (std: 20.912)
Parameters: {'max_features': 4, 'n_estimators': np.int64(150), 'max_depth': 17}

Model with rank: 3
Mean validation score: 806.846 (lower is better) (std: 21.831)
Parameters: {'max_features': 4, 'n_estimators': np.int64(140), 'max_depth': 17}


Best Parameters found by Pattern Search:
{'max_features': 4, 'n_estimators': np.int64(130), 'max_depth': 17}

Best Score (mean_absolute_error):
805.038061 (lower is better)

DataFrame of Evaluated Points (Pattern Search):
    max_features  n_estimators  max_depth  mean_absolute_error
0              3           130         11          1128.966638
1              4           130       

In [9]:
# metrics for the prototype (bugs intact)
old_elapsed = time.time() - t_old_start
old_n_fits = len(pattern_search.ResultDf)
old_best = {k: (int(v) if isinstance(v, (np.integer, float)) and float(v).is_integer()
                else v)
            for k, v in pattern_search.best_params_.items()}
old_mae = abs(pattern_search.best_score_)

cvres_old = cross_validate(make_clf().set_params(**old_best), X, y, cv=tscv,
                           scoring=("r2", "neg_mean_absolute_error"), n_jobs=-1)
old_r2 = cvres_old["test_r2"].mean()
print("OLD  best:", old_best)
print(f"OLD  fits={old_n_fits}  equiv={float(old_n_fits):.2f}  time={old_elapsed:.1f}s")
print(f"OLD  CV MAE={old_mae:.3f}  CV R2={old_r2:.6f}")


OLD  best: {'max_features': 4, 'n_estimators': 130, 'max_depth': 17}
OLD  fits=17  equiv=17.00  time=1546.2s
OLD  CV MAE=805.038  CV R2=0.809692


In [10]:
# ============ HEAD-TO-HEAD ============
print("=" * 76)
print("HEAD-TO-HEAD - 26-value grid, this machine, same sklearn 1.9, MAE objective")
print("=" * 76)
hdr = f"{'':24s}{'OLD (prototype, bugs)':>24s}{'NEW (PatternSearchCV)':>24s}"
print(hdr)
print(f"{'evaluations':24s}{old_n_fits:>24d}{new_n_fits:>24d}")
print(f"{'full-fit equivalents':24s}{float(old_n_fits):>24.2f}{new_equiv:>24.2f}")
print(f"{'wall-clock (s)':24s}{old_elapsed:>24.1f}{new_elapsed:>24.1f}")
print(f"{'best point':24s}{str(tuple(old_best.values())):>24s}{str(tuple(new_best.values())):>24s}")
print(f"{'CV MAE (best)':24s}{old_mae:>24.3f}{new_mae:>24.3f}")
print(f"{'CV R2  (best)':24s}{old_r2:>24.6f}{new_r2:>24.6f}")
print()
print(f"speedup (wall-clock): {old_elapsed / new_elapsed:.2f}x")
print(f"compute ratio (equivalents): {float(old_n_fits) / new_equiv:.2f}x")


HEAD-TO-HEAD - 26-value grid, this machine, same sklearn 1.9, MAE objective
                           OLD (prototype, bugs)   NEW (PatternSearchCV)
evaluations                                   17                      23
full-fit equivalents                       17.00                    6.80
wall-clock (s)                            1546.2                   824.1
best point                          (4, 130, 17)            (4, 150, 17)
CV MAE (best)                            805.038                 805.730
CV R2  (best)                           0.809692                0.809981

speedup (wall-clock): 1.88x
compute ratio (equivalents): 2.50x
